# Notebook 11.1  An instruction-routing and honest-evaluation harness

*One model, many tasks, and the per-dialect table and audit the chapter argues for.*

---

*Companion notebook to* **Introduction to Arabic Speech Technologies** *by Hend S. Al-Khalifa.*

**How to run.** Open this notebook in Google Colab or run it locally with the
pinned environment in `requirements.txt`. Every notebook in this series runs end
to end with **no downloads and no accounts**: where a real corpus or a
pretrained model is unavailable, a clearly marked fallback stands in for it, and
the notebook says which path it took. Cells that need a download are marked
`OPTIONAL` and are safe to skip.

**On data.** Where you substitute a real corpus, record its release version and
its licence in the provenance cell at the end. A result without them is not
reproducible, which is the habit this book asks for in every chapter.

<a href="https://colab.research.google.com/github/arabic-speech-book/arabic-speech-book.github.io/blob/main/docs/notebooks/ch11_audio_llm_prompting.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## What this notebook does

An audio-language model serves many tasks through one interface: you give it
audio and an instruction, and it transcribes, translates, identifies a dialect
or answers a question depending on what you asked. This notebook builds the
harness around that idea, which is the part a reader can build today and the
part the chapter argues is usually built badly.

1. Route free-text instructions, in English and in Arabic, to tasks.
2. Score each task with its own metric, because one number over four tasks is meaningless.
3. Report every result per dialect.
4. Audit the evaluation itself for the four shortcuts that inflate Arabic scores: labels generated by a model, synthesized test audio, overlap between training and test, and a test set with one speaker standing in for a dialect.

The model here is a mock. Everything around it is real, and the mock is one
function call away from a real audio-language model.

In [ ]:
NOTEBOOK = 'ch11_audio_llm_prompting.ipynb'

import re
from collections import defaultdict

import numpy as np

rng = np.random.default_rng(3)
TASKS = ['transcribe', 'translate', 'identify_dialect', 'answer']
print('tasks:', ', '.join(TASKS))

## 1. The clips, with the metadata an audit needs

Each clip carries more than audio and a label. It carries the dialect, the
speaker, where the label came from, and whether the audio is real or
synthesized. Those last two fields are the ones that make an evaluation
auditable, and they are the ones most datasets do not have.

In [ ]:
CLIPS = [
    {'id': 'c1', 'dialect': 'MSA', 'speaker': 's1', 'audio': 'real',
     'label_source': 'human',
     'transcript': 'الطقس حار اليوم', 'english': 'the weather is hot today',
     'question': 'ما هو الطقس', 'answer': 'حار'},
    {'id': 'c2', 'dialect': 'Gulf', 'speaker': 's2', 'audio': 'real',
     'label_source': 'human',
     'transcript': 'وين اقرب صيدلية', 'english': 'where is the nearest pharmacy',
     'question': 'ماذا يطلب المتحدث', 'answer': 'صيدلية'},
    {'id': 'c3', 'dialect': 'Egyptian', 'speaker': 's3', 'audio': 'real',
     'label_source': 'model',
     'transcript': 'عايز اروح المطار', 'english': 'i want to go to the airport',
     'question': 'الى اين', 'answer': 'المطار'},
    {'id': 'c4', 'dialect': 'Maghrebi', 'speaker': 's4', 'audio': 'synthesized',
     'label_source': 'human',
     'transcript': 'بغيت نمشي للدار', 'english': 'i want to go home',
     'question': 'الى اين', 'answer': 'الدار'},
    {'id': 'c5', 'dialect': 'Gulf', 'speaker': 's2', 'audio': 'real',
     'label_source': 'human',
     'transcript': 'ابغى احجز موعد', 'english': 'i want to book an appointment',
     'question': 'ماذا يريد', 'answer': 'موعد'},
]
TRAIN_IDS = {'c3'}          # a clip that was also in the training set
print(f'{len(CLIPS)} clips, dialects '
      f'{sorted({c["dialect"] for c in CLIPS})}')

## 2. Instruction routing

An audio-language model does not have a task argument. It has an instruction,
and the task is whatever the instruction asked for. So the first thing to build
is the router, and the first thing to notice is that it must work in Arabic as
well as English, because your users will write in Arabic.

In [ ]:
ROUTES = [
    ('transcribe', ['transcribe', 'write down', 'what was said',
                    'اكتب', 'فرغ', 'ماذا قال']),
    ('translate', ['translate', 'in english', 'ترجم', 'بالانجليزية']),
    ('identify_dialect', ['which dialect', 'identify the dialect', 'accent',
                          'اي لهجة', 'حدد اللهجة']),
    ('answer', ['answer', 'question', 'اجب', 'سؤال']),
]


def route(instruction):
    text = instruction.lower()
    for task, cues in ROUTES:
        if any(cue in text for cue in cues):
            return task
    return 'unknown'


for instruction in ['Transcribe this clip.',
                    'ترجم هذا المقطع بالانجليزية',
                    'Which dialect is this speaker using?',
                    'اجب عن السؤال التالي',
                    'Summarize the mood of the speaker.']:
    print(f'  {route(instruction):>16}  <-  {instruction}')
print('\nThe last one routes to unknown, and that is the correct behaviour. '
      'A router that guesses a task for every instruction will silently score '
      'the wrong metric on it.')

## 3. A mock model, and four metrics

The mock produces output of the right shape with a controllable error rate, so
the harness can be tested before a real model is attached. Replace `mock_model`
with a call to a real audio-language model and nothing else changes.

Each task gets its own metric: word error rate for transcription, BLEU for
translation, accuracy for dialect identification, exact match for the question.
A single average over the four would be a number with no units.

In [ ]:
def mock_model(clip, task, error_rate=0.25):
    # stands in for a real ALM: right most of the time, wrong in ways that
    # look like the model rather than like noise
    wrong = rng.random() < error_rate
    if task == 'transcribe':
        words = clip['transcript'].split()
        if wrong and len(words) > 1:
            words[rng.integers(len(words))] = 'كلمة'
        return ' '.join(words)
    if task == 'translate':
        return clip['english'] if not wrong else \
            clip['english'].replace('the', 'a')
    if task == 'identify_dialect':
        others = [d for d in {c['dialect'] for c in CLIPS}
                  if d != clip['dialect']]
        return clip['dialect'] if not wrong else str(rng.choice(others))
    if task == 'answer':
        return clip['answer'] if not wrong else 'لا اعرف'
    return ''


def wer(reference, hypothesis):
    r, h = reference.split(), hypothesis.split()
    d = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    d[:, 0] = np.arange(len(r) + 1)
    d[0, :] = np.arange(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1,
                          d[i - 1, j - 1] + (r[i - 1] != h[j - 1]))
    return d[-1, -1] / max(1, len(r))


def unigram_bleu(reference, hypothesis):
    r, h = reference.split(), hypothesis.split()
    if not h:
        return 0.0
    overlap = sum(min(h.count(w), r.count(w)) for w in set(h))
    return overlap / len(h)


METRICS = {
    'transcribe': ('WER, lower is better', lambda c, o: wer(c['transcript'], o)),
    'translate': ('unigram BLEU, higher is better',
                  lambda c, o: unigram_bleu(c['english'], o)),
    'identify_dialect': ('accuracy, higher is better',
                         lambda c, o: float(o == c['dialect'])),
    'answer': ('exact match, higher is better',
               lambda c, o: float(o.strip() == c['answer'])),
}

results = []
for clip in CLIPS:
    for task in TASKS:
        output = mock_model(clip, task)
        _, metric = METRICS[task]
        results.append({'clip': clip['id'], 'dialect': clip['dialect'],
                        'task': task, 'output': output,
                        'score': metric(clip, output)})

print(f'{"task":>18}  {"pooled":>8}   metric')
for task in TASKS:
    scores = [r['score'] for r in results if r['task'] == task]
    print(f'{task:>18}  {np.mean(scores):8.3f}   {METRICS[task][0]}')

## 4. The same results, per dialect

This is the table the chapter argues for. The pooled row above is what most
papers print. The table below is what a reader needs, and the two tell
different stories whenever a model is better at Modern Standard Arabic than at
anything else, which is almost always.

In [ ]:
dialects = sorted({c['dialect'] for c in CLIPS})
print(f'{"task":>18} ' + ' '.join(f'{d:>10}' for d in dialects) +
      f' {"pooled":>8} {"spread":>8}')
for task in TASKS:
    row = []
    for d in dialects:
        scores = [r['score'] for r in results
                  if r['task'] == task and r['dialect'] == d]
        row.append(np.mean(scores) if scores else np.nan)
    pooled = np.nanmean(row)
    spread = np.nanmax(row) - np.nanmin(row)
    print(f'{task:>18} ' + ' '.join(f'{v:10.3f}' for v in row) +
          f' {pooled:8.3f} {spread:8.3f}')
print('\nThe spread column is the one to read first. A small pooled number '
      'with a large spread is a system that works for some speakers of '
      'Arabic and not others, and the pooled number alone will never say so.')

## 5. The audit

Four shortcuts inflate Arabic evaluation scores, and all four are invisible in
the results table. Each one is a property of the test set, not of the model, so
the only way to catch them is to check the test set.

Run this before you believe your own numbers.

In [ ]:
def audit(clips, train_ids):
    findings = []

    model_labelled = [c['id'] for c in clips if c['label_source'] != 'human']
    if model_labelled:
        findings.append(
            f'labels generated by a model: {model_labelled}. A model scored '
            f'against another model\'s labels is measuring agreement, not '
            f'accuracy.')

    synthetic = [c['id'] for c in clips if c['audio'] != 'real']
    if synthetic:
        findings.append(
            f'synthesized test audio: {synthetic}. Synthetic speech is '
            f'cleaner and more regular than the speech a system will meet, '
            f'so this inflates every score computed on it.')

    overlap = sorted({c['id'] for c in clips} & set(train_ids))
    if overlap:
        findings.append(
            f'clips in both training and test: {overlap}. Every number that '
            f'includes them is contaminated, and the contamination cannot be '
            f'subtracted out afterwards.')

    speakers = defaultdict(set)
    for c in clips:
        speakers[c['dialect']].add(c['speaker'])
    thin = [d for d, s in speakers.items() if len(s) < 2]
    if thin:
        findings.append(
            f'a dialect represented by one speaker: {thin}. That row of the '
            f'per-dialect table is a per-speaker result wearing a dialect '
            f'label.')

    return findings


findings = audit(CLIPS, TRAIN_IDS)
print(f'{len(findings)} finding(s):\n')
for i, f in enumerate(findings, 1):
    print(f'{i}. {f}\n')
if not findings:
    print('none, which is rarer than it sounds. Check the audit itself.')

## 6. Where a real model goes

The harness above is complete. What it lacks is a model. Any audio-language
model that takes audio plus an instruction fits into `mock_model`'s place: pass
the waveform and the instruction, take back the text.

Two cautions from the chapter, and they survive the model swap:

- the instruction is part of the experiment. The same model with a different prompt is a different system, so record the exact instruction strings beside the results;
- capability is rising fast, and the evaluation is not keeping up. When a model does well on this harness, the useful next question is not whether it can do the task but what in the test set let it look that good.

In [ ]:
RUN_REAL = False

if RUN_REAL:
    # sketch: any model exposing audio plus instruction as one call
    # from transformers import pipeline
    # alm = pipeline('automatic-speech-recognition', model='<an ALM>')
    # output = alm({'array': waveform, 'sampling_rate': 16000},
    #              generate_kwargs={'prompt': instruction})
    pass
else:
    print('skipped: attach a real audio-language model here')

INSTRUCTIONS_USED = {
    'transcribe': 'Transcribe this clip.',
    'translate': 'ترجم هذا المقطع بالانجليزية',
    'identify_dialect': 'Which dialect is this speaker using?',
    'answer': 'اجب عن السؤال التالي',
}
print('\nrecord these with the results:')
for task, text in INSTRUCTIONS_USED.items():
    print(f'  {task:>18}: {text}')

## Provenance

Fill this in before you quote any number from this notebook. It is the same
information the chapter's Reproducibility Note asks for, and it is the
difference between a result and a screenshot.

In [ ]:
PROVENANCE = {
    'notebook': NOTEBOOK,
    'ran_on': 'fill in the date you ran it',
    'data': 'corpus name and release version, or "synthetic fallback"',
    'licence': 'the licence of the data you used',
    'model': 'model name and revision, or "none"',
    'normalization': 'the normalization applied before scoring',
    'hardware': 'CPU or the GPU model',
}
for k, v in PROVENANCE.items():
    print(f'{k:>15}: {v}')